# M2b 평가지표와 교차검증 — 실습 (W4, M2 2부작 완결편)

> ⚠️ **가장 먼저 — 화면 위 [Drive로 복사]를 누르세요.**
> 지금 보고 있는 것은 원본을 잠깐 띄운 **임시 사본**입니다. 복사하지 않으면 탭을 닫는 순간
> 채운 빈칸과 실행 결과가 **모두 사라집니다.** 복사본은 내 Google Drive에 저장되고,
> 원본은 바뀌지 않으니 마음껏 고쳐도 됩니다.

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. 혼동행렬 **손계산**(0.93 / 0.615 / 0.80 / 0.696)을 sklearn과 대조한다 ⭐
2. 유방암 실전 성적표를 읽는다 — 정확도 0.942인데 **놓친 환자 9명**(재현율 0.859)
3. **정확도의 함정**을 재현한다 — "무조건 정상"이 0.945
4. 회귀 지표를 손계산으로 검증한다 — **R² = −0.75**의 의미까지
5. **분할 운**(0.906~0.971)을 실측하고 **교차검증**(0.951±0.018)으로 이긴다

**7단계 멘탈모델 초점:** 평가(Evaluation)

## Part A. 혼동행렬 손계산 검증 ⭐
100명 검진(환자 10·정상 90), 모델 결과 **TP=8, FN=2, FP=5, TN=85**.
**먼저 종이에서** 네 지표를 완주한 뒤(§4의 표), sklearn과 대조하세요.

In [ ]:
import numpy as np                                     # 수치 계산
from sklearn.metrics import (confusion_matrix, accuracy_score,   # 성적표 도구들
                             precision_score, recall_score, f1_score)

y_true = np.array([1]*10 + [0]*90)                     # 실제: 환자 10, 정상 90
y_pred = np.array([1]*8 + [0]*2 + [1]*5 + [0]*85)      # 예측: TP8·FN2·FP5·TN85가 되도록
print(confusion_matrix(y_true, y_pred))                # [[TN FP], [FN TP]] 순서 주의!

acc = accuracy_score(y_true, y_pred)                   # (8+85)/100
pre = precision_score(y_true, y_pred)                  # 8/13
rec = recall_score(y_true, y_pred)                     # 8/10
f1 = ___ * pre * rec / (pre + rec)                     # ✍️ 빈칸: 조화평균 공식의 분자 계수
print('정확도:', acc, '| 정밀도:', round(pre, 4),
      '| 재현율:', rec, '| F1:', round(f1, 4))         # 손계산과 일치?

> **손계산 검증 완료:** 0.93 / 0.6154 / 0.8 / 0.6957 — §4의 손계산 그대로. sklearn의 혼동행렬은 **[[TN, FP], [FN, TP]]** 순서(교재 그림과 배치가 다름 — 실무 단골 함정). F1의 `2`는 조화평균의 계수 — **낮은 쪽에 끌려가는** 평균(정밀도 1.0·재현율 0.01이면 F1≈0.02).

## Part B. 실전 — 유방암 진단 성적표
M2a와 같은 데이터에 로지스틱 회귀(M5에서 배울 모델 — 지금은 "분류기")를 학습시켜 성적표를 읽습니다.
⚠️ **함정 먼저:** sklearn 유방암 데이터의 원래 라벨은 **1 = benign(양성 종양 = 정상 쪽)** 입니다. "1 = 환자(악성)"가 되도록 **뒤집고 시작**합니다 — 지표를 읽기 전에 "1이 무엇인지"부터 확인하는 것이 실무의 첫 습관.

In [ ]:
from sklearn.datasets import load_breast_cancer        # 유방암 데이터
from sklearn.model_selection import train_test_split    # 분할(M2a)
from sklearn.linear_model import LogisticRegression     # 분류기
import matplotlib.pyplot as plt                         # 그래프

data = load_breast_cancer()
print('원래 라벨:', list(data.target_names))             # ['malignant', 'benign'] — 1=benign(정상 쪽)!
X = data.data
y = 1 - data.target                                      # 뒤집기: 1 = 악성(환자) — 지표의 "양성"을 환자로
X_train, X_test, y_train, y_test = train_test_split(     # M2a와 동일 방식 분할
    X, y, test_size=0.3, random_state=42, stratify=y)

model = LogisticRegression(max_iter=5000)                # 분류기(반복 넉넉히)
model.fit(X_train, y_train)                              # train으로 학습
y_pred = model.predict(X_test)                           # 봉인된 시험지 채점

print(confusion_matrix(y_test, y_pred))                  # 성적표는 행렬부터
rec = recall_score(___, ___)                             # ✍️ 빈칸: 재현율 = (정답, 예측) 순서
print('정확도:', round(accuracy_score(y_test, y_pred), 3),
      '| 정밀도:', round(precision_score(y_test, y_pred), 3),
      '| 재현율:', round(rec, 3),
      '| F1:', round(f1_score(y_test, y_pred), 3))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay      # 혼동행렬 시각화
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test,  # 모델·시험셋으로 바로 그림
                                      display_labels=['normal (0)', 'patient (1)'])
plt.title('Confusion matrix (breast cancer, 1=malignant)')  # 제목(영어)
plt.show()

> 혼동행렬 [[106, 1], [9, 55]] — 시험셋 환자(악성) 64명 중 55명을 잡고 **9명을 놓쳤습니다**(재현율 **0.859**). 정확도 0.942로 훌륭해 보이지만, 암 진단이라면 "놓친 9명"이 헤드라인. 정밀도는 0.982(오경보 단 1명) — **깐깐한 판정 = 정밀도↑·재현율↓** 상충의 실전 예시이기도 합니다. 읽는 순서: ①행렬 → ②비용 구조 → ③주 지표.

## Part C. 정확도의 함정 — 불균형 데이터
양성 5%뿐인 데이터에서 **"무조건 정상"** 모델의 성적표를 봅니다.

In [ ]:
from sklearn.datasets import make_classification        # 인공 불균형 데이터
from sklearn.dummy import DummyClassifier                # 규칙 기반 더미 모델

Xi, yi = make_classification(n_samples=2000, weights=[0.95, 0.05],  # 음성 95% / 양성 5%
                             random_state=0)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    Xi, yi, test_size=0.3, random_state=0, stratify=yi)  # 비율 유지(M2a stratify!)

dummy = DummyClassifier(strategy='___')                  # ✍️ 빈칸: "가장 흔한 클래스"로만 답하는 전략
dummy.fit(Xi_tr, yi_tr)
yp = dummy.predict(Xi_te)                                # 전부 다수(음성)로 예측
print('무조건 정상 — 정확도:', round(accuracy_score(yi_te, yp), 3),
      '| 재현율:', round(recall_score(yi_te, yp), 3))    # 0.945 / 0.0 ?!

lr = LogisticRegression(max_iter=5000).fit(Xi_tr, yi_tr) # 진짜 모델과 비교
yp2 = lr.predict(Xi_te)
print('로지스틱   — 정확도:', round(accuracy_score(yi_te, yp2), 3),
      '| 재현율:', round(recall_score(yi_te, yp2), 3),
      '| 정밀도:', round(precision_score(yi_te, yp2), 3))

> **정확도 0.945인데 재현율 0** — 환자를 한 명도 못 잡는 모델이 94.5점. 불균형의 기준선 = **다수 클래스 비율.** 진짜 모델(재현율 0.879)과의 차이는 정확도가 아니라 **재현율**에서 드러납니다. (이 혼동행렬은 2학기 D1의 MNIST에서 10×10으로 재등장)

## Part D. 회귀 지표 — 손계산 검증
정답 [3, 5, 7], 예측 [4, 3, 10] → 오차 [+1, −2, +3]. **먼저 종이에서** MAE·MSE·RMSE·R²를 완주하세요.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_t = np.array([3.0, 5.0, 7.0])                          # 정답
y_p = np.array([4.0, 3.0, 10.0])                         # 예측(오차 +1, -2, +3)

mae = mean_absolute_error(y_t, y_p)                      # (1+2+3)/3
mse = mean_squared_error(y_t, y_p)                       # (1+4+9)/3
rmse = mse ** ___                                        # ✍️ 빈칸: 제곱근 = 몇 제곱?
r2 = r2_score(y_t, y_p)                                  # 1 - 14/8
print('MAE:', mae, '| MSE:', round(mse, 4),
      '| RMSE:', round(rmse, 4), '| R2:', r2)            # 손계산과 일치?

> **MAE 2.0 / MSE 4.67 / RMSE 2.16 / R² = −0.75.** 음수 R²는 버그가 아닙니다 — "평균만 찍는 모델"(제곱오차합 8)보다 우리 모델(14)이 **못했다**는 정당한 경고(1−14/8). MSE의 "제곱 = 큰 오차를 크게 벌함"은 M4의 손실 함수로 재등장합니다.

## Part E. 분할 운, 그리고 교차검증 ⭐
같은 데이터·같은 모델로 random_state만 바꾸면 시험 점수가 얼마나 출렁일까요?

In [ ]:
from sklearn.model_selection import cross_val_score      # 교차검증

lucks = []
for rs in range(10):                                      # 분할 10가지
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=rs, stratify=y)
    m = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
    lucks.append(m.score(Xte, yte))                       # 그때그때의 시험 점수
print('한 번 분할 10회:', np.round(lucks, 3))
print('폭:', round(max(lucks) - min(lucks), 3))           # 0.064 — 분할 운!

scores = cross_val_score(model, X, y, cv=___)             # ✍️ 빈칸: 몇 겹으로?
print('교차검증 점수:', np.round(scores, 3))
print('평균:', round(scores.mean(), 3), '± 표준편차:', round(scores.std(), 3))  # 0.951 ± 0.018

plt.scatter(range(10), lucks, color='#dc2626', label='single split (10 random states)')
plt.axhline(scores.mean(), color='#2563eb', label=f'5-fold CV mean = {scores.mean():.3f}')
plt.fill_between([-0.5, 9.5], scores.mean()-scores.std(), scores.mean()+scores.std(),
                 color='#2563eb', alpha=0.15, label='CV mean +/- std')
plt.xlabel('random_state'); plt.ylabel('test accuracy')   # 축(영어)
plt.title('Split luck vs cross-validation'); plt.legend(fontsize=8)
plt.xlim(-0.5, 9.5); plt.tight_layout(); plt.show()

> 한 번 분할은 **0.906~0.971로 출렁**(폭 0.064) — 교차검증은 **0.951 ± 0.018**이라는 정직한 한 문장을 줍니다. 용도 정리: **튜닝·모델 선택 = 교차검증**(반복해도 test 오염 없음 — M2a "반복 튜닝 금지"의 해법), **test = 최종 발표 한 번.**

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "TP8·FN2·FP5·TN85의 네 지표를 내가 완주해 볼 테니 채점해 줘."
- "정밀도 1.0·재현율 0.01의 F1을 계산해 볼게 — 조화평균이 왜 정직한지 논해 줘."
- "암 진단/스팸 필터/얼굴 인식 잠금해제 — 각각의 주 지표를 내가 고르고 이유를 댈게."
- "R² = −0.75가 나왔어 — 버그인지 판정하는 절차를 같이 세워 줘."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 혼동행렬 손계산(0.93/0.615/0.80/0.696)을 sklearn과 대조했다 — sklearn 행렬은 [[TN,FP],[FN,TP]]!
2. 라벨 뒤집기 함정을 넘고 유방암 성적표(놓친 9명)를 읽었다 · 정확도의 함정(0.945, 재현율 0)과 R²=−0.75의 의미를 검증했다
3. 분할 운(폭 0.064)을 실측하고 교차검증(0.951±0.018)으로 이겼다

**스스로 점검**
- [ ] 네 지표를 네 칸에서 손으로 계산할 수 있다
- [ ] "놓침(FN)이 아픈 문제"와 "오경보(FP)가 아픈 문제"의 예를 하나씩 댈 수 있다
- [ ] 지표를 읽기 전 "1이 무엇인지"부터 확인하는 습관이 생겼다
- [ ] 불균형의 기준선(다수 비율)과 비교하는 습관이 생겼다
- [ ] 튜닝은 교차검증, test는 마지막 한 번 — 이유를 안다

**🔹심화 (선택)**
- **임계값 실험:** `model.predict_proba(X_test)[:, 1] > t`로 t를 0.3/0.5/0.7로 바꿔 정밀도·재현율의 상충을 실측하세요(암 진단이라면 t를 어느 쪽으로?).
- **StratifiedKFold를 직접**: `cross_val_score`가 분류에서 자동으로 쓰는 것을 명시적으로 만들어 비교.
- **cv를 3/5/10으로**: 평균·표준편차가 어떻게 변하나 — 겹 수의 트레이드오프.

**다음 시간(M3):** 첫 모델 KNN — 오늘 만든 자(분할·지표·교차검증)로 잽니다. 스케일링이 결과를 뒤집는 드라마 예고.